In [93]:
print("hello")

hello


In [94]:
from dotenv import load_dotenv
load_dotenv()

True

In [95]:
import pandas as pd

# QA
inputs = [
    "For customer-facing applications, which company's models dominate the top rankings?",
    "What percentage of respondents are using RAG in some form?",
    "How often are most respondents updating their models?",
]

outputs = [
    "OpenAI models dominate, with 3 of the top 5 and half of the top 10 most popular models for customer-facing apps.",
    "70% of respondents are using RAG in some form.",
    "More than 50% update their models at least monthly, with 17% doing so weekly.",
]

# Dataset
qa_pairs = [{"question": q, "answer": a} for q, a in zip(inputs, outputs)]
df = pd.DataFrame(qa_pairs)

# Write to csv
csv_path = "C:/Users/kasar/LLMOPS_series/data/goldens.csv"
df.to_csv(csv_path, index=False)

In [96]:
from langsmith import Client

client = Client()
dataset_name = "llmops_dataset"

# Store
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Input and expected output pairs for AgenticAIReport",
)
client.create_examples(
    inputs=[{"question": q} for q in inputs],
    outputs=[{"answer": a} for a in outputs],
    dataset_id=dataset.id,
)

{'example_ids': ['fd440776-7fe6-4efb-9e75-2ca473bc4719',
  'f9118c32-e645-4274-b0b7-7564618bdae8',
  'e206e449-1981-466d-8770-3bc21e32b22b'],
 'count': 3,
 'as_of': '2026-03-09T00:40:38.762765506Z'}

In [98]:
import sys
sys.path.append("C:/Users/kasar/LLMOPS_series")

from pathlib import Path
from multi_doc_chat.src.document_ingestion.data_ingestion import ChatIngestor
from multi_doc_chat.src.document_chat.retrieval import ConversationalRAG
import os

# Simple file adapter for local file paths
class LocalFileAdapter:
    """Adapter for local file paths to work with ChatIngestor."""
    def __init__(self, file_path: str):
        self.path = Path(file_path)
        self.name = self.path.name
    
    def getbuffer(self) -> bytes:
        return self.path.read_bytes()


def answer_ai_report_question(
    inputs: dict,
    data_path: str = "C:/Users/kasar/LLMOPS_series/data/The 2025 AI Engineering Report.txt",
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
    k: int = 5
) -> dict:
    """
    Answer questions about the AI Engineering Report using RAG.
    
    Args:
        inputs: Dictionary containing the question, e.g., {"question": "What is RAG?"}
        data_path: Path to the AI Engineering Report text file
        chunk_size: Size of text chunks for splitting
        chunk_overlap: Overlap between chunks
        k: Number of documents to retrieve
    
    Returns:
        Dictionary with the answer, e.g., {"answer": "RAG stands for..."}
    """
    try:
        # Extract question from inputs
        question = inputs.get("question", "")
        if not question:
            return {"answer": "No question provided"}
        
        # Check if file exists
        if not Path(data_path).exists():
            return {"answer": f"Data file not found: {data_path}"}
        
        # Create file adapter
        file_adapter = LocalFileAdapter(data_path)
        
        # Build index using ChatIngestor
        ingestor = ChatIngestor(
            temp_base="data",
            faiss_base="faiss_index",
            use_session_dirs=True
        )
        
        # Build retriever
        ingestor.built_retriver(
            uploaded_files=[file_adapter],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            k=k
        )
        
        # Get session ID and index path
        session_id = ingestor.session_id
        index_path = f"faiss_index/{session_id}"
        
        # Create RAG instance and load retriever
        rag = ConversationalRAG(session_id=session_id)
        rag.load_retriever_from_faiss(
            index_path=index_path,
            k=k,
            index_name=os.getenv("FAISS_INDEX_NAME", "index")
        )
        
        # Get answer
        answer = rag.invoke(question, chat_history=[])
        
        return {"answer": answer}
        
    except Exception as e:
        return {"answer": f"Error: {str(e)}"}

In [99]:
# Test the function with a sample question
test_input = {"question": "What is the “AI expectation–execution gap” and why does it occur according to the 2025 AI Engineering Report?"}
result = answer_ai_report_question(test_input)
print("Question:", test_input["question"])
print("\nAnswer:", result["answer"])

{"timestamp": "2026-03-09T00:40:47.637785Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-03-09T00:40:47.639010Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-03-09T00:40:47.639905Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_2v...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2026-03-09T00:40:47.640785Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-03-09T00:40:47.644989Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260308_194047_d1273d41", "temp_dir": "data\\session_20260308_194047_d1273d41", "faiss_dir": "faiss_index\\session_20260308_194047_d1273d41", "sessionized": true, "timestamp": "2026-03-09T00:40:47.647139Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "The 2025 AI Engineering Report.txt", "saved_

Question: What is the “AI expectation–execution gap” and why does it occur according to the 2025 AI Engineering Report?

Answer: The “AI expectation–execution gap” refers to the gap between AI experimentation and real-world production impact. It occurs because organizations often underestimate the complexity of building production-ready AI systems, which require robust infrastructure, reliable data pipelines, governance frameworks, and continuous monitoring systems.


In [100]:
from langsmith.evaluation import evaluate

In [101]:
# Compatibility wrapper so your existing code can stay almost the same
class LangChainStringEvaluator:
    def __init__(self, evaluator_type):
        self.evaluator_type = evaluator_type

    def __call__(self, run, example):
        actual_output = run.outputs.get("answer", "")
        expected_output = example.outputs.get("answer", "")
        input_question = example.inputs.get("question", "")

        from langchain_google_genai import ChatGoogleGenerativeAI
        from langchain_core.prompts import ChatPromptTemplate

        eval_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an evaluator whose job is to judge question-answering quality.

Evaluate the answer based on correctness, semantic similarity, and completeness.
Do not penalize wording differences if the meaning is the same.

Respond with:
1. A brief reasoning (1-2 sentences)
2. A final verdict: either "CORRECT" or "INCORRECT"

Format your response as:
Reasoning: [your reasoning]
Verdict: [CORRECT or INCORRECT]"""),
            ("human", """<example>
<input>
{input}
</input>

<output>
Expected Output: {expected_output}

Actual Output: {actual_output}
</output>
</example>""")
        ])

        llm = ChatGoogleGenerativeAI(
            model="gemini-2.5-pro",
            temperature=0
        )

        chain = eval_prompt | llm

        try:
            response = chain.invoke({
                "input": input_question,
                "expected_output": expected_output,
                "actual_output": actual_output
            })

            response_text = response.content

            reasoning = ""
            verdict = ""

            for line in response_text.split("\n"):
                if line.startswith("Reasoning:"):
                    reasoning = line.replace("Reasoning:", "").strip()
                elif line.startswith("Verdict:"):
                    verdict = line.replace("Verdict:", "").strip()

            score = 1 if verdict.upper() == "CORRECT" else 0

            return {
                "key": self.evaluator_type,
                "score": score,
                "comment": f"Reasoning: {reasoning} | Verdict: {verdict}"
            }

        except Exception as e:
            return {
                "key": self.evaluator_type,
                "score": 0,
                "comment": f"Error during evaluation: {str(e)}"
            }

In [102]:
# Example: Test with all golden questions
print("Testing all questions from the dataset:\n")
for i, q in enumerate(inputs, 1):
    test_input = {"question": q}
    result = answer_ai_report_question(test_input)
    print(f"Q{i}: {q}")
    print(f"A{i}: {result['answer']}\n")
    print("-" * 80 + "\n")


{"timestamp": "2026-03-09T00:41:38.818980Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-03-09T00:41:38.820302Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-03-09T00:41:38.821619Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_2v...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2026-03-09T00:41:38.822519Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-03-09T00:41:38.825454Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260308_194138_19ef31cd", "temp_dir": "data\\session_20260308_194138_19ef31cd", "faiss_dir": "faiss_index\\session_20260308_194138_19ef31cd", "sessionized": true, "timestamp": "2026-03-09T00:41:38.827749Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "The 2025 AI Engineering Report.txt", "saved_

Testing all questions from the dataset:



{"added": 1, "index": "faiss_index\\session_20260308_194138_19ef31cd", "timestamp": "2026-03-09T00:41:39.506328Z", "level": "info", "event": "FAISS index updated"}
{"k": 5, "fetch_k": 20, "lambda_mult": 0.5, "timestamp": "2026-03-09T00:41:39.507185Z", "level": "info", "event": "Using MMR search"}
{"timestamp": "2026-03-09T00:41:39.509676Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-03-09T00:41:39.510293Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-03-09T00:41:39.511197Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_2v...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2026-03-09T00:41:39.511942Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-03-09T00:41:39.513759Z", "level": "info", "event": "YAML config loaded"}
{"provider": "google", "model": "gemini-2.0-fl

Q1: For customer-facing applications, which company's models dominate the top rankings?
A1: I'm sorry, but the provided text does not contain information about which company's models dominate the top rankings for customer-facing applications.

--------------------------------------------------------------------------------



{"added": 1, "index": "faiss_index\\session_20260308_194141_9dd26e3e", "timestamp": "2026-03-09T00:41:41.930012Z", "level": "info", "event": "FAISS index updated"}
{"k": 5, "fetch_k": 20, "lambda_mult": 0.5, "timestamp": "2026-03-09T00:41:41.931395Z", "level": "info", "event": "Using MMR search"}
{"timestamp": "2026-03-09T00:41:41.934992Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-03-09T00:41:41.937082Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-03-09T00:41:41.938453Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_2v...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2026-03-09T00:41:41.939104Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-03-09T00:41:41.941492Z", "level": "info", "event": "YAML config loaded"}
{"provider": "google", "model": "gemini-2.0-fl

Q2: What percentage of respondents are using RAG in some form?
A2: I don't know.

--------------------------------------------------------------------------------



{"added": 1, "index": "faiss_index\\session_20260308_194143_c3e9d30a", "timestamp": "2026-03-09T00:41:44.171631Z", "level": "info", "event": "FAISS index updated"}
{"k": 5, "fetch_k": 20, "lambda_mult": 0.5, "timestamp": "2026-03-09T00:41:44.172404Z", "level": "info", "event": "Using MMR search"}
{"timestamp": "2026-03-09T00:41:44.174575Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-03-09T00:41:44.175003Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-03-09T00:41:44.175392Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_2v...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2026-03-09T00:41:44.176203Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-03-09T00:41:44.178021Z", "level": "info", "event": "YAML config loaded"}
{"provider": "google", "model": "gemini-2.0-fl

Q3: How often are most respondents updating their models?
A3: I'm sorry, but the provided text does not contain information about how often respondents are updating their models.

--------------------------------------------------------------------------------



In [104]:
from langsmith.evaluation import evaluate

# Evaluators
qa_evaluator = [LangChainStringEvaluator("cot_qa")]
dataset_name = "llmops_dataset"

# Run evaluation using our RAG function
experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=qa_evaluator,
    experiment_prefix="test-llmops_dataset-qa-rag",
    # Experiment metadata
    metadata={
        "variant": "RAG with FAISS and AI Engineering Report",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)

View the evaluation results for experiment: 'test-llmops_dataset-qa-rag-4b6343d3' at:
https://smith.langchain.com/o/57bc4e8f-c602-4818-ad0a-cd2cd07d0e51/datasets/e74546cd-4db1-4ab8-885b-4a4191d2ebf8/compare?selectedSessions=b921ce1a-96d5-40d6-a351-c3bb1d9f8483




0it [00:00, ?it/s]{"timestamp": "2026-03-09T00:43:56.600616Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-03-09T00:43:56.601838Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-03-09T00:43:56.603114Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_2v...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2026-03-09T00:43:56.604114Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-03-09T00:43:56.610619Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260308_194356_69558e8b", "temp_dir": "data\\session_20260308_194356_69558e8b", "faiss_dir": "faiss_index\\session_20260308_194356_69558e8b", "sessionized": true, "timestamp": "2026-03-09T00:43:56.614324Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "The 2025 AI Engineering Re

Custom Correctness Evaluator

In [105]:
from langsmith.schemas import Run, Example
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

def correctness_evaluator(run: Run, example: Example) -> dict:
    """
    Custom LLM-as-a-Judge evaluator for correctness.
    
    Correctness means how well the actual model output matches the reference output 
    in terms of factual accuracy, coverage, and meaning.
    
    Args:
        run: The Run object containing the actual outputs
        example: The Example object containing the expected outputs
    
    Returns:
        dict with 'score' (1 for correct, 0 for incorrect) and 'comment'
    """
    # Extract actual and expected outputs
    actual_output = run.outputs.get("answer", "")
    expected_output = example.outputs.get("answer", "")
    input_question = example.inputs.get("question", "")
    
    # Define the evaluation prompt
    eval_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an evaluator whose job is to judge correctness.

Correctness means how well the actual model output matches the reference output in terms of factual accuracy, coverage, and meaning.

- If the actual output matches the reference output semantically (even if wording differs), it should be marked correct.
- If the output misses key facts, introduces contradictions, or is factually incorrect, it should be marked incorrect.

Do not penalize for stylistic or formatting differences unless they change meaning."""),
        ("human", """<example>
<input>
{input}
</input>

<output>
Expected Output: {expected_output}

Actual Output: {actual_output}
</output>
</example>

Please grade the following agent run given the input, expected output, and actual output.
Focus only on correctness (semantic and factual alignment).

Respond with:
1. A brief reasoning (1-2 sentences)
2. A final verdict: either "CORRECT" or "INCORRECT"

Format your response as:
Reasoning: [your reasoning]
Verdict: [CORRECT or INCORRECT]""")
    ])
    
    # Initialize LLM (using Gemini as shown in your config)
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-pro",
        temperature=0
    )
    
    # Create chain and invoke
    chain = eval_prompt | llm
    
    try:
        response = chain.invoke({
            "input": input_question,
            "expected_output": expected_output,
            "actual_output": actual_output
        })
        
        response_text = response.content
        
        # Parse the response
        reasoning = ""
        verdict = ""
        
        for line in response_text.split('\n'):
            if line.startswith("Reasoning:"):
                reasoning = line.replace("Reasoning:", "").strip()
            elif line.startswith("Verdict:"):
                verdict = line.replace("Verdict:", "").strip()
        
        # Convert verdict to score (1 for correct, 0 for incorrect)
        score = 1 if verdict.upper() == "CORRECT" else 0
        
        return {
            "key": "correctness",
            "score": score,
            "comment": f"Reasoning: {reasoning} | Verdict: {verdict}"
        }
        
    except Exception as e:
        return {
            "key": "correctness",
            "score": 0,
            "comment": f"Error during evaluation: {str(e)}"
        }

Run Evaluation with Custom Correctness Evaluator

In [107]:
# Run evaluation with the custom correctness evaluator
from langsmith.evaluation import evaluate

# Define evaluators - using custom correctness evaluator
evaluators = [correctness_evaluator]

dataset_name = "llmops_dataset"

# Run evaluation
experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=evaluators,
    experiment_prefix="agenticAIReport-correctness-eval",
    description="Evaluating RAG system with custom correctness evaluator (LLM-as-a-Judge)",
    metadata={
        "variant": "RAG with FAISS and AI Engineering Report",
        "evaluator": "custom_correctness_llm_judge",
        "model": "gemini-2.5-pro",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)

print("\nEvaluation completed! Check the LangSmith UI for detailed results.")


View the evaluation results for experiment: 'agenticAIReport-correctness-eval-65e7fa53' at:
https://smith.langchain.com/o/57bc4e8f-c602-4818-ad0a-cd2cd07d0e51/datasets/e74546cd-4db1-4ab8-885b-4a4191d2ebf8/compare?selectedSessions=6aed9e2d-a5e2-4f06-bd6c-1faa81a215ec




0it [00:00, ?it/s]{"timestamp": "2026-03-09T00:45:16.866781Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-03-09T00:45:16.867984Z", "level": "info", "event": "Loaded GROQ_API_KEY from individual env var"}
{"timestamp": "2026-03-09T00:45:16.869288Z", "level": "info", "event": "Loaded GOOGLE_API_KEY from individual env var"}
{"keys": {"GROQ_API_KEY": "gsk_2v...", "GOOGLE_API_KEY": "AIzaSy..."}, "timestamp": "2026-03-09T00:45:16.871095Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["embedding_model", "retriever", "llm"], "timestamp": "2026-03-09T00:45:16.873943Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260308_194516_8fb859ca", "temp_dir": "data\\session_20260308_194516_8fb859ca", "faiss_dir": "faiss_index\\session_20260308_194516_8fb859ca", "sessionized": true, "timestamp": "2026-03-09T00:45:16.876181Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "The 2025 AI Engineering Re


Evaluation completed! Check the LangSmith UI for detailed results.


Optional: Combine Multiple Evaluators